# 03 — Silver Cleaning

This notebook reads from Bronze and produces the **Silver layer** — clean, standardised, business-ready data.

What we do here:
- Remove duplicate transactions
- Cast columns to the correct data types
- Standardise merchant names and categories
- Add date part columns (year, month, day, hour) for easy filtering
- Add SA-specific flags: `is_month_end`, `is_salary_period`, `is_weekend`
- Filter out invalid rows (null IDs, zero amounts)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, TimestampType

BRONZE_PATH = 'dbfs:/FileStore/sa-banking-pipeline/bronze'
SILVER_PATH = 'dbfs:/FileStore/sa-banking-pipeline/silver'

print('Spark ready:', spark.version)

## Step 1 — Read Bronze tables

In [ ]:
customers_bronze = spark.read.format('delta').load(f'{BRONZE_PATH}/customers')
txn_bronze       = spark.read.format('delta').load(f'{BRONZE_PATH}/transactions')

print(f'Bronze customers: {customers_bronze.count():,} rows')
print(f'Bronze transactions: {txn_bronze.count():,} rows')

## Step 2 — Clean customers table

In [ ]:
customers_silver = customers_bronze \
    .dropDuplicates(['customer_id']) \
    .withColumn('province', F.initcap(F.trim(F.col('province')))) \
    .withColumn('segment', F.lower(F.trim(F.col('segment')))) \
    .withColumn('monthly_income', F.col('monthly_income').cast(DoubleType())) \
    .filter(F.col('customer_id').isNotNull()) \
    .filter(F.col('monthly_income') > 0) \
    .withColumn('_silver_processed_at', F.current_timestamp())

print(f'Silver customers: {customers_silver.count():,} rows')
customers_silver.show(5)

## Step 3 — Clean transactions table

This is where most of the work happens. We add SA-specific flags that will be used in the Gold layer to build spending intelligence features.

In [ ]:
txn_silver = txn_bronze \
    .dropDuplicates(['transaction_id']) \
    .withColumn('amount', F.col('amount').cast(DoubleType())) \
    .withColumn('transaction_date', F.col('transaction_date').cast(TimestampType())) \
    .withColumn('merchant_name', F.upper(F.trim(F.col('merchant_name')))) \
    .withColumn('merchant_category', F.lower(F.trim(F.col('merchant_category')))) \
    .withColumn('transaction_type', F.upper(F.trim(F.col('transaction_type')))) \
    .withColumn('channel', F.upper(F.trim(F.col('channel')))) \
    .filter(F.col('amount') > 0) \
    .filter(F.col('transaction_id').isNotNull()) \
    .filter(F.col('customer_id').isNotNull())

print(f'After dedup and filter: {txn_silver.count():,} rows')

In [ ]:
# Add date part columns
txn_silver = txn_silver \
    .withColumn('txn_year',        F.year('transaction_date')) \
    .withColumn('txn_month',       F.month('transaction_date')) \
    .withColumn('txn_day',         F.dayofmonth('transaction_date')) \
    .withColumn('txn_day_of_week', F.dayofweek('transaction_date')) \
    .withColumn('txn_hour',        F.hour('transaction_date')) \
    .withColumn('is_weekend',      F.dayofweek('transaction_date').isin([1, 7]))

# SA-specific flags
txn_silver = txn_silver \
    .withColumn('is_month_end',    F.col('txn_day') >= 20) \
    .withColumn('is_salary_period', F.col('txn_day').between(24, 26)) \
    .withColumn('_silver_processed_at', F.current_timestamp())

print('SA flags added: is_month_end, is_salary_period, is_weekend')
txn_silver.select('transaction_id','amount','txn_day','is_month_end','is_salary_period','is_weekend').show(10)

## Step 4 — Write Silver Delta tables

In [ ]:
# Write customers Silver
customers_silver.write.format('delta').mode('overwrite').save(f'{SILVER_PATH}/customers')
print(f'✅ Silver customers written')

# Write transactions Silver — partition by year and month
txn_silver.write.format('delta').mode('overwrite') \
    .partitionBy('txn_year', 'txn_month') \
    .save(f'{SILVER_PATH}/transactions')
print(f'✅ Silver transactions written (partitioned by year/month)')

## Step 5 — Data quality checks

In [ ]:
# Null check on key columns
key_cols = ['transaction_id', 'customer_id', 'amount', 'transaction_date', 'merchant_category']
null_counts = txn_silver.select([
    F.sum(F.col(c).isNull().cast('int')).alias(c) for c in key_cols
])
print('Null counts (should all be 0):')
null_counts.show()

# Amount range check
print('Amount stats (ZAR):')
txn_silver.select('amount').describe().show()

## ✅ Done

Data is now clean, typed, and enriched with SA-specific flags in the Silver layer.

**Next:** Open `04_gold_features.ipynb`